# SigLIP embedding extraction

Takes the `crops/` folder (from `generate_synthetic_crops.py` for now, real player crops once YOLO is ready) and turns each image into an embedding vector using a pretrained SigLIP model. No fine-tuning needed here.

**Run cells top to bottom.** This is a one-off batch job, not a live session — run it, download the output, close the tab.

In [ ]:
!pip install -q transformers torch pillow pandas

## 1. Get the crops into Colab

Two options — use whichever matches your setup:

**Option A — zip upload (simplest for now):** on your laptop, zip the `crops/` folder (which includes `manifest.csv`) and upload it below.

**Option B — clone from your repo:** if `crops/` is committed to the repo, uncomment and run the `git clone` cell instead.

In [ ]:
# Option A: upload crops.zip (a zip of your crops/ folder, manifest.csv included)
from google.colab import files
uploaded = files.upload()  # select crops.zip

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("crops")

# If the zip contained a top-level "crops" folder, flatten it so manifest.csv is at crops/manifest.csv
if os.path.exists("crops/crops/manifest.csv") and not os.path.exists("crops/manifest.csv"):
    import shutil
    for f in os.listdir("crops/crops"):
        shutil.move(f"crops/crops/{f}", f"crops/{f}")

print("Contents of crops/:", os.listdir("crops")[:10])

In [ ]:
# Option B: clone from repo instead (uncomment and edit REPO_URL)
# REPO_URL = "https://github.com/your-org/tacticeye.git"
# !git clone $REPO_URL repo
# !cp -r repo/crops ./crops

## 2. Load SigLIP and extract embeddings

In [ ]:
import torch
from transformers import AutoProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "google/siglip-base-patch16-224"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

In [ ]:
import pandas as pd
from PIL import Image
import numpy as np

manifest = pd.read_csv("crops/manifest.csv")
print(f"{len(manifest)} crops to embed")

BATCH_SIZE = 16
all_embeddings = []
all_track_ids = []

for start in range(0, len(manifest), BATCH_SIZE):
    batch = manifest.iloc[start:start + BATCH_SIZE]
    images = [Image.open(f"crops/{p}").convert("RGB") for p in batch["crop_path"]]
    inputs = processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():
        feats = model.get_image_features(**inputs)
    feats = feats.cpu().numpy()
    all_embeddings.append(feats)
    all_track_ids.extend(batch["track_id"].tolist())
    print(f"Embedded {start + len(batch)}/{len(manifest)}")

embeddings = np.concatenate(all_embeddings, axis=0)
print("Embeddings shape:", embeddings.shape)

## 3. Save output — download this file, it's the artifact this notebook exists to produce

In [ ]:
out_df = pd.DataFrame(embeddings, columns=[f"e{i}" for i in range(embeddings.shape[1])])
out_df.insert(0, "track_id", all_track_ids)
out_df.to_parquet("embeddings.parquet", index=False)

from google.colab import files
files.download("embeddings.parquet")
print("Saved and downloading embeddings.parquet - run cluster_teams.py on this locally.")